# Combine the train, val and test CSV files

This notebook combines the three split files of each task into one CSV file.

| Input files | Output file |
|---|---|
| `split_age_train.csv`, `split_age_test.csv`, `split_age_val.csv` | `split_age.csv` |
| `split_gender_train.csv`, `split_gender_test.csv`, `split_gender_val.csv` | `split_gender.csv` |
| `split_real_fake_train.csv`, `split_real_fake_test.csv`, `split_real_fake_val.csv` | `split_real_fake.csv` |

Run `06_create_csv_from_split_folders.ipynb` first, because it creates the input files.

The combined file keeps the same columns as the input files (`image_id` and the class column).
A combined file does not show which images belong to train, val or test.
If you want that information, set `ADD_SPLIT_COLUMN = True` in Step 1 and a `split` column will be added.

In [4]:
import pandas as pd
from pathlib import Path

## Step 1: Set the paths

In [5]:
DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")

# task name -> name of the class column in its CSV files
TASKS = {
    "split_age":       "age",
    "split_gender":    "gender",
    "split_real_fake": "label",
}
SPLITS = ["train", "test", "val"]      # the order in which the files are joined

ADD_SPLIT_COLUMN = False               # True = also add a column that says train / val / test

## Step 2: Combine the files

For each task, this step:

1. reads the train, test and val CSV files
2. joins them into one table
3. sorts the table by image id
4. saves it as one CSV file (for example `split_age.csv`)

It also checks that the number of rows is correct and that no image id appears twice.

In [6]:
combined = {}

for task, column_name in TASKS.items():
    parts = []
    for split in SPLITS:
        file_path = DATA_DIR / f"{task}_{split}.csv"
        if not file_path.exists():
            raise FileNotFoundError("File not found: " + str(file_path) + ". Run notebook 06 first.")

        part = pd.read_csv(file_path, dtype=str)       # read everything as text so ids like 007 stay unchanged
        print(f"{file_path.name:28s} rows = {len(part):5d}")
        if ADD_SPLIT_COLUMN:
            part["split"] = split
        parts.append(part)

    df = pd.concat(parts, ignore_index=True)

    # sort by image id (numbers first, then text ids)
    df["sort_key"] = pd.to_numeric(df["image_id"], errors="coerce")
    df = df.sort_values(["sort_key", "image_id"]).drop(columns="sort_key").reset_index(drop=True)

    # checks
    expected_rows = sum(len(p) for p in parts)
    repeated = df["image_id"].duplicated().sum()
    print(f"   combined rows = {len(df)} (expected {expected_rows})")
    if len(df) != expected_rows:
        print("   WARNING: the row count is not correct")
    if repeated > 0:
        print(f"   WARNING: {repeated} image ids appear more than once (the same image is in two splits or two classes)")

    out_path = DATA_DIR / f"{task}.csv"
    df.to_csv(out_path, index=False)
    print("   saved:", out_path.name, "\n")

    combined[task] = df

split_age_train.csv          rows =  3561
split_age_test.csv           rows =   764
split_age_val.csv            rows =   763
   combined rows = 5088 (expected 5088)
   saved: split_age.csv 

split_gender_train.csv       rows =  3561
split_gender_test.csv        rows =   764
split_gender_val.csv         rows =   763
   combined rows = 5088 (expected 5088)
   saved: split_gender.csv 

split_real_fake_train.csv    rows =  3561
split_real_fake_test.csv     rows =   764
split_real_fake_val.csv      rows =   763
   combined rows = 5088 (expected 5088)
   saved: split_real_fake.csv 



## Step 3: Check the number of images in each class

In [7]:
for task, column_name in TASKS.items():
    print("=====", task, "=====")
    counts = combined[task][column_name].value_counts()
    print(counts)
    print("TOTAL:", counts.sum(), "\n")

===== split_age =====
age
not_applicable    2638
26-35             1484
36-50              338
18-25              330
50+                219
under_18            79
Name: count, dtype: int64
TOTAL: 5088 

===== split_gender =====
gender
unknown    2612
female     1453
male       1023
Name: count, dtype: int64
TOTAL: 5088 

===== split_real_fake =====
label
real    2593
fake    2495
Name: count, dtype: int64
TOTAL: 5088 



## Step 4: Look at the combined files

In [8]:
combined["split_age"].head(10)

,image_id,age
0,1,26-35
1,2,under_18
2,3,26-35
3,4,26-35
4,5,36-50
5,6,18-25
6,7,50+
7,8,18-25
8,9,36-50
9,10,not_applicable


In [9]:
combined["split_gender"].head(10)

,image_id,gender
0,1,male
1,2,male
2,3,male
3,4,male
4,5,male
5,6,male
6,7,female
7,8,female
8,9,male
9,10,female


In [10]:
combined["split_real_fake"].head(10)

,image_id,label
0,1,real
1,2,real
2,3,real
3,4,real
4,5,real
5,6,real
6,7,real
7,8,real
8,9,real
9,10,real
